In [47]:
import pandas as pd

file_path = "/Users/vijeethvj8/Downloads/GEN2/WORK/week1/sales_data.xlsx"

df = pd.read_excel(file_path)
                 
print("Raw shape:", df.shape)
print(df.head())

Raw shape: (48, 11)
   OrderID  OrderDate CustomerID CustomerName Region   Product  \
0     1001 2026-03-01       C001   John Smith   West    Laptop   
1     1002 2026-03-01       C002  Maria Lopez   East     Mouse   
2     1003 2026-03-02       C003    David Lee  South  Keyboard   
3     1004 2026-03-02       C004   Sarah Khan  North   Monitor   
4     1005 2026-03-03       C005  Chris Evans   West   Printer   

          Category  Quantity  UnitPrice SalesRep  SalesAmount  
0         Hardware         2      850.0    Alice       1700.0  
1      Accessories         5       25.0      Bob        125.0  
2      Accessories         3       45.0    Carol        135.0  
3         Hardware         2      220.0    Alice        440.0  
4  Office Supplies         1      180.0      Bob        180.0  


In [49]:
# Standardize column names
df.columns = [col.strip() for col in df.columns]

# Trim whitespace from string columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# Replace blank-like strings with NaN
df = df.replace(["", "nan", "None"], np.nan)

# Fix dates
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")

# Convert numeric columns
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Fill missing CustomerName with 'Unknown'
df["CustomerName"] = df["CustomerName"].fillna("Unknown")

# Remove rows with invalid dates
df = df.dropna(subset=["OrderDate"])

# Fill missing Quantity with median
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

# Fill missing UnitPrice with median
df["UnitPrice"] = df["UnitPrice"].fillna(df["UnitPrice"].median())

# Remove rows with impossible values
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# Convert quantity to integer
df["Quantity"] = df["Quantity"].round().astype(int)

# Create SalesAmount column
df["SalesAmount"] = df["Quantity"] * df["UnitPrice"]

# Remove duplicate rows if any
df = df.drop_duplicates()

# Sort data
df = df.sort_values(by=["OrderDate", "OrderID"])

# Save cleaned dataset
df.to_csv("sales_cleaned.csv", index=False)

print("\nCleaned shape:", df.shape)
print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nSummary:")
print("Total revenue:", round(df["SalesAmount"].sum(), 2))
print("Top region:", df.groupby("Region")["SalesAmount"].sum().idxmax())
print("Top category:", df.groupby("Category")["SalesAmount"].sum().idxmax())


Cleaned shape: (48, 11)

Missing values after cleaning:
OrderID         0
OrderDate       0
CustomerID      0
CustomerName    0
Region          0
Product         0
Category        0
Quantity        0
UnitPrice       0
SalesRep        0
SalesAmount     0
dtype: int64

Summary:
Total revenue: 17175.5
Top region: East
Top category: Hardware


In [51]:
import sqlite3
import pandas as pd

In [53]:
conn = sqlite3.connect("sales.db")

In [55]:
df.to_sql("sales_data", conn, if_exists="replace", index=False)

48

In [57]:
query = "SELECT * FROM sales_data LIMIT 5"
result = pd.read_sql(query, conn)

print(result)

   OrderID            OrderDate CustomerID CustomerName Region   Product  \
0     1001  2026-03-01 00:00:00       C001   John Smith   West    Laptop   
1     1002  2026-03-01 00:00:00       C002  Maria Lopez   East     Mouse   
2     1003  2026-03-02 00:00:00       C003    David Lee  South  Keyboard   
3     1004  2026-03-02 00:00:00       C004   Sarah Khan  North   Monitor   
4     1005  2026-03-03 00:00:00       C005  Chris Evans   West   Printer   

          Category  Quantity  UnitPrice SalesRep  SalesAmount  
0         Hardware         2      850.0    Alice       1700.0  
1      Accessories         5       25.0      Bob        125.0  
2      Accessories         3       45.0    Carol        135.0  
3         Hardware         2      220.0    Alice        440.0  
4  Office Supplies         1      180.0      Bob        180.0  


In [59]:
# GROUP BY
query = """
SELECT Region, SUM(SalesAmount) AS TotalRevenue
FROM sales_data
GROUP BY Region
"""
pd.read_sql(query, conn)

,Region,TotalRevenue
0,East,5236.0
1,North,5001.0
2,South,3388.0
3,West,3550.5


In [61]:
# Subquery
query = """
SELECT *
FROM sales_data
WHERE SalesAmount > (
    SELECT AVG(SalesAmount) FROM sales_data
)
"""
pd.read_sql(query, conn)

,OrderID,OrderDate,CustomerID,CustomerName,Region,Product,Category,Quantity,UnitPrice,SalesRep,SalesAmount
0,1001,2026-03-01 00:00:00,C001,John Smith,West,Laptop,Hardware,2,850.0,Alice,1700.0
1,1004,2026-03-02 00:00:00,C004,Sarah Khan,North,Monitor,Hardware,2,220.0,Alice,440.0
2,1006,2026-03-03 00:00:00,C006,Nina Patel,East,Laptop,Hardware,1,900.0,Carol,900.0
3,1012,2026-03-06 00:00:00,C012,Sophia Hall,North,Laptop,Hardware,2,870.0,Carol,1740.0
4,1017,2026-03-09 00:00:00,C017,Ben Carter,West,Monitor,Hardware,2,230.0,Bob,460.0
5,1019,2026-03-10 00:00:00,C019,Henry Walker,South,Laptop,Hardware,1,910.0,Alice,910.0
6,1026,2026-03-13 00:00:00,C026,Abigail Ward,East,Laptop,Hardware,2,880.0,Bob,1760.0
7,1034,2026-03-17 00:00:00,C034,Unknown,North,Monitor,Hardware,2,215.0,Alice,430.0
8,1036,2026-03-18 00:00:00,C036,Nina Patel,East,Laptop,Hardware,1,920.0,Carol,920.0
9,1042,2026-03-21 00:00:00,C042,Sophia Hall,North,Laptop,Hardware,2,875.0,Carol,1750.0
